<a href="https://colab.research.google.com/github/manisha17marndi-lgtm/ecommerce-funnel-retention-analytics./blob/main/notebooks/funnel_retention_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='ga4-funnel-project-515805')

users = client.query("""
    SELECT * FROM `ga4-funnel-project-515805.ga4_project.rpt_user_metrics`
""").to_dataframe()

print(users.shape)
users.head()

(257299, 19)


,user_id,first_visit_date,cohort_week,channel_group,device_category,eligible_d1,eligible_d7_and_funnel,eligible_d30_and_repeat,did_view_item_7d,did_add_to_cart_7d,did_begin_checkout_7d,orders_7d,revenue_7d,orders_30d,ret_d1_session,ret_d7_session,ret_d30_session,ret_d1_shopping,ret_d7_shopping
0,79772400.2663581547,2020-11-01,2020-11-01,Data deleted / other,desktop,True,True,True,True,False,True,0,0.0,0,False,False,False,False,False
1,3969608.9768981542,2020-11-03,2020-11-01,Data deleted / other,mobile,True,True,True,False,False,False,0,0.0,0,True,False,False,False,False
2,18134304.4702227145,2020-11-03,2020-11-01,Data deleted / other,desktop,True,True,True,False,False,False,0,0.0,0,False,False,False,False,False
3,28501628.5968300908,2020-11-03,2020-11-01,Data deleted / other,desktop,True,True,True,False,False,False,0,0.0,0,False,False,False,False,False
4,72247217.2012519479,2020-11-04,2020-11-01,Data deleted / other,desktop,True,True,True,False,False,False,0,0.0,0,False,False,False,False,False


In [3]:
print("shape:", users.shape)
print("distinct users:", users['user_id'].nunique())

f = users[users['eligible_d7_and_funnel']]
print("funnel cohort:", len(f))
print("viewed item:", int(f['did_view_item_7d'].sum()))
print("added to cart:", int(f['did_add_to_cart_7d'].sum()))
print("began checkout:", int(f['did_begin_checkout_7d'].sum()))
print("buyers (7d):", int((f['orders_7d'] > 0).sum()))
print("revenue (7d):", round(f['revenue_7d'].sum(), 2))

print("D1 returned:", int(users.loc[users['eligible_d1'], 'ret_d1_session'].sum()))
print("D7 returned:", int(f['ret_d7_session'].sum()))
print("D30 returned:", int(users.loc[users['eligible_d30_and_repeat'], 'ret_d30_session'].sum()))

shape: (257299, 19)
distinct users: 257299
funnel cohort: 238636
viewed item: 50110
added to cart: 9644
began checkout: 7231
buyers (7d): 2424
revenue (7d): 185106.0
D1 returned: 10754
D7 returned: 1586
D30 returned: 223


In [4]:
from datetime import date

raw = client.query("""
    SELECT user_pseudo_id AS user_id,
           TIMESTAMP_MICROS(event_timestamp) AS ts,
           ecommerce.transaction_id AS order_id,
           ecommerce.purchase_revenue_in_usd AS revenue
    FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
    WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
      AND event_name = 'purchase'
""").to_dataframe()

fv = client.query("""
    SELECT user_id, first_visit_ts, had_earlier_activity
    FROM `ga4-funnel-project-515805.ga4_project.users_with_channel`
""").to_dataframe()

print("raw purchase events (all users):", len(raw))

p = raw.merge(fv, on='user_id', how='inner')
print("purchase events for users in users table:", len(p))

valid = p[p['order_id'].notna() & (p['order_id'] != '(not set)') & (p['revenue'] > 0)]
orders_py = (valid.sort_values('ts')
                  .drop_duplicates(['order_id', 'user_id'], keep='first'))

print("valid orders:", len(orders_py))
print("buyers:", orders_py['user_id'].nunique())
print("revenue:", round(orders_py['revenue'].sum(), 2))

cut = pd.Timestamp('2021-01-24', tz='UTC')
o7 = orders_py[
    (~orders_py['had_earlier_activity'])
    & (orders_py['first_visit_ts'].dt.normalize() <= cut)
    & (orders_py['ts'] >= orders_py['first_visit_ts'])
    & (orders_py['ts'] <= orders_py['first_visit_ts'] + pd.Timedelta(days=7))
]
print("7-day orders:", len(o7))
print("7-day buyers:", o7['user_id'].nunique())
print("7-day revenue:", round(o7['revenue'].sum(), 2))

raw purchase events (all users): 5692
purchase events for users in users table: 4409
valid orders: 3542
buyers: 3007
revenue: 243430.0
7-day orders: 2739
7-day buyers: 2424
7-day revenue: 185106.0


In [5]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.multitest import multipletests

users['cohort_week'] = pd.to_datetime(users['cohort_week'])
users['bought_7d'] = users['orders_7d'] > 0
f = users[users['eligible_d7_and_funnel']].copy()

org = f[f['channel_group'] == 'Organic Search']
paid = f[f['channel_group'] == 'Paid Search']
store = f[f['channel_group'] == 'Store referral (own domain)']
desk = f[f['device_category'] == 'desktop']
mob = f[f['device_category'] == 'mobile']
tab = f[f['device_category'] == 'tablet']

d1 = users[users['eligible_d1']]
d7 = users[users['eligible_d7_and_funnel']]
early = lambda d: d[d['cohort_week'] <= '2020-11-29']
holi = lambda d: d[d['cohort_week'].isin(pd.to_datetime(['2020-12-13', '2020-12-20', '2020-12-27']))]

tests = [
    ("Checkout to purchase: Store referral vs Organic",
     store[store['did_begin_checkout_7d']]['bought_7d'], org[org['did_begin_checkout_7d']]['bought_7d']),
    ("View to cart: Paid Search vs Organic",
     paid[paid['did_view_item_7d']]['did_add_to_cart_7d'], org[org['did_view_item_7d']]['did_add_to_cart_7d']),
    ("Purchased in 7 days: Paid Search vs Organic", paid['bought_7d'], org['bought_7d']),
    ("Purchased in 7 days: Desktop vs Mobile", desk['bought_7d'], mob['bought_7d']),
    ("Purchased in 7 days: Tablet vs Desktop", tab['bought_7d'], desk['bought_7d']),
    ("Day 1 return: Nov cohorts vs mid/late-Dec cohorts",
     early(d1)['ret_d1_session'].astype(bool), holi(d1)['ret_d1_session'].astype(bool)),
    ("Day 7 return: Nov cohorts vs mid/late-Dec cohorts",
     early(d7)['ret_d7_session'].astype(bool), holi(d7)['ret_d7_session'].astype(bool)),
    ("Day 7 return: Store referral vs Organic",
     store['ret_d7_session'].astype(bool), org['ret_d7_session'].astype(bool)),
    ("Day 1 return: Paid Search vs Organic",
     d1[d1['channel_group'] == 'Paid Search']['ret_d1_session'].astype(bool),
     d1[d1['channel_group'] == 'Organic Search']['ret_d1_session'].astype(bool)),
]

rows = []
for name, a, b in tests:
    xa, na, xb, nb = int(a.sum()), len(a), int(b.sum()), len(b)
    z, p = proportions_ztest([xa, xb], [na, nb])
    la, ha = proportion_confint(xa, na, method='wilson')
    lb, hb = proportion_confint(xb, nb, method='wilson')
    rows.append({
        'test': name,
        'rate_a_%': round(100 * xa / na, 2), 'n_a': na,
        'ci_a_%': f"{100*la:.2f}-{100*ha:.2f}",
        'rate_b_%': round(100 * xb / nb, 2), 'n_b': nb,
        'ci_b_%': f"{100*lb:.2f}-{100*hb:.2f}",
        'diff_pp': round(100 * (xa / na - xb / nb), 2),
        'z': round(z, 2), 'p_raw': p
    })

res = pd.DataFrame(rows)
res['p_holm'] = multipletests(res['p_raw'], method='holm')[1]
res['p_raw'] = res['p_raw'].round(4)
res['p_holm'] = res['p_holm'].round(4)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)
print(res.to_string(index=False))

ch = users[['user_id', 'channel_group']]
o7c = o7.merge(ch, on='user_id')
a = o7c.loc[o7c['channel_group'] == 'Paid Search', 'revenue'].values
b = o7c.loc[o7c['channel_group'] == 'Organic Search', 'revenue'].values
rng = np.random.default_rng(42)
diffs = np.array([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(10000)])
print("\nPaid orders:", len(a), "Organic orders:", len(b))
print("Mean order value Paid:", round(a.mean(), 2), "Organic:", round(b.mean(), 2))
print("Bootstrap 95% CI for difference:", round(np.percentile(diffs, 2.5), 2), "to", round(np.percentile(diffs, 97.5), 2))

                                             test  rate_a_%    n_a      ci_a_%  rate_b_%    n_b      ci_b_%  diff_pp     z  p_raw  p_holm
  Checkout to purchase: Store referral vs Organic     28.35    381 24.05-33.07     34.04   2782 32.30-35.82    -5.69 -2.21 0.0271  0.1894
             View to cart: Paid Search vs Organic     17.63   2705 16.24-19.12     18.77  19279 18.23-19.33    -1.14 -1.42 0.1547  0.7009
      Purchased in 7 days: Paid Search vs Organic      1.02  13049   0.86-1.21      1.03  91545   0.97-1.10    -0.02 -0.16 0.8721  1.0000
           Purchased in 7 days: Desktop vs Mobile      1.01 138616   0.96-1.06      1.04  94670   0.97-1.10    -0.03 -0.73 0.4654  1.0000
           Purchased in 7 days: Tablet vs Desktop      0.88   5350   0.66-1.17      1.01 138616   0.96-1.06    -0.13 -0.92 0.3567  1.0000
Day 1 return: Nov cohorts vs mid/late-Dec cohorts      5.16  87461   5.02-5.31      3.32  58145   3.18-3.47     1.84 16.72 0.0000  0.0000
Day 7 return: Nov cohorts vs mid/l